测试编解码器

In [1]:
from wingVAE import WingVAE
import numpy as np
import torch
import os
from general_generation_system import AircraftScriptGenerator
from WingScriptGenerator import param_to_ggs_wing_dict, calculate_full_wing_aerodynamic_char
import math
generator = AircraftScriptGenerator()

In [2]:
from wingVAE import MEAN, STD, SCALE_MEAN, SCALE_STD, apply_norm, denormalize_geometry

In [3]:
model = WingVAE(
        input_dim=46, 
        latent_dim=32, 
        hidden_dim=256, 
        condition_dim=3,
        switch_epoch=200, 
        kl_coeff=0.005, 
        phys_coeff=1.5
    )

#加载模型权重
model_path = "D:/generate_3/Wing_generation/exp/Encoder_exp21"
checkpoint = torch.load(os.path.join(model_path, "last.ckpt"), map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['state_dict'])
model.eval()


C:\Users\WednZ\AppData\Local\Temp\ipykernel_30036\2418434171.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(model_path, "last.ckpt

WingVAE(
  (encoder_backbone): Sequential(
    (0): Linear(in_features=46, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): ReLU()
  )
  (fc_mu): Linear(in_features=256, out_features=32, bias=True)
  (fc_var): Linear(in_features=256, out_features=32, bias=True)
  (decoder_geometry): Sequential(
    (0): Linear(in_features=32, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=46, bias=True)
  )
  (decoder_physics): Sequential(
    (0): Linear(in_features=35, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [4]:
wingPartParam = np.array([0.154051125,0.099343032,0.262970291,0.067577599,0.303525766,0.170648694,0.164297604,0.203076205,0.055550936,-0.024332408,-0.154051125,-0.099343032,-0.262970291,-0.067577599,-0.303525766,-0.170648694,-0.164297604,-0.203076205,-0.055550936,0.024332408,0.148912039,0.115814941,0.268338306,0.075926679,0.294531817,0.192806848,0.187872426,0.19226277,0.164500801,-0.0006764,-0.159190211,-0.082871124,-0.257602276,-0.059228519,-0.312519716,-0.148490541,-0.140722783,-0.213889641,0.05339893,0.047988416,1.0, 1.0,0.616512384,0.787345211,-0.054917934,0.998490872])#最后6个值对应lg(\lambda), lg(\bar b), \Lambda(sin, cos), \epsilon(sin,cos)
normalizedWingPartParam = apply_norm(wingPartParam)
normalizedWingPartParam = torch.tensor(normalizedWingPartParam, dtype=torch.float)

In [5]:
mu = model.encode(normalizedWingPartParam[:].float().unsqueeze(0))  # 添加 batch 维度
print("Encoded latent mean (mu):", mu)
decoded_output = model.decode_geometry(mu)
scaled_decoded_geometry = decoded_output[0, :].detach().numpy()  # 获取几何参数部分
restored_geometry = denormalize_geometry(scaled_decoded_geometry)
print("Restored geometry parameters:", restored_geometry)

Encoded latent mean (mu): tensor([[ 0.0414, -0.9051, -0.9139,  2.0341,  0.9183,  0.7523,  0.9486,  0.1930,
         -0.8117, -1.2415, -0.0193, -0.6258, -0.5248, -0.3614, -1.2664,  0.5912,
          0.3718,  0.6683, -0.3067, -0.9428, -0.1842,  0.1302,  0.5932, -0.1211,
         -0.1362, -1.2864, -0.6471, -0.2866,  0.1808,  0.4521,  1.0212, -0.5718]],
       grad_fn=<AddmmBackward0>)
Restored geometry parameters: [ 0.1532418   0.09868219  0.26480185  0.07205285  0.28372713  0.18494856
  0.14797659  0.2020598   0.05754977 -0.02355165 -0.15236259 -0.09643475
 -0.25763797 -0.07147227 -0.31049717 -0.17500869 -0.16342197 -0.20451161
 -0.05534772  0.02576266  0.14631668  0.11170002  0.25710246  0.08583507
  0.28637329  0.20387904  0.15801172  0.19039019  0.1508833   0.0039457
 -0.15510601 -0.07895654 -0.26487227 -0.05034569 -0.30873268 -0.14615706
 -0.15753137 -0.21436569  0.04628787  0.05221678  1.02915239  0.98871267
  0.6179388   0.79206194 -0.04917231  1.0024896 ]


In [6]:
wingPartParam

array([ 1.54051125e-01,  9.93430320e-02,  2.62970291e-01,  6.75775990e-02,
        3.03525766e-01,  1.70648694e-01,  1.64297604e-01,  2.03076205e-01,
        5.55509360e-02, -2.43324080e-02, -1.54051125e-01, -9.93430320e-02,
       -2.62970291e-01, -6.75775990e-02, -3.03525766e-01, -1.70648694e-01,
       -1.64297604e-01, -2.03076205e-01, -5.55509360e-02,  2.43324080e-02,
        1.48912039e-01,  1.15814941e-01,  2.68338306e-01,  7.59266790e-02,
        2.94531817e-01,  1.92806848e-01,  1.87872426e-01,  1.92262770e-01,
        1.64500801e-01, -6.76400000e-04, -1.59190211e-01, -8.28711240e-02,
       -2.57602276e-01, -5.92285190e-02, -3.12519716e-01, -1.48490541e-01,
       -1.40722783e-01, -2.13889641e-01,  5.33989300e-02,  4.79884160e-02,
        1.00000000e+00,  1.00000000e+00,  6.16512384e-01,  7.87345211e-01,
       -5.49179340e-02,  9.98490872e-01])

In [31]:
random_z = torch.randn(1, 32)  # 生成一个随机的潜空间向量
print("Random latent vector (z):", random_z)
# random_z = torch.tensor([0]*32, dtype=torch.float).unsqueeze(0)  # 使用全1的潜空间向量进行测试
generated_geometry = model.decode_geometry(random_z)
restored_generated_geometry = denormalize_geometry(generated_geometry[0, :].detach().numpy())
print("Generated geometry parameters from random latent vector:", restored_generated_geometry)

Random latent vector (z): tensor([[ 0.2837,  0.4793, -1.3349, -0.1527, -0.1267, -1.0278,  0.7341, -1.5324,
         -3.2115,  1.0753, -0.7597,  1.9351, -0.3169,  0.4886, -1.9090,  0.1158,
         -0.5697, -0.2498,  0.7899, -0.8105,  0.9360,  0.2217,  0.5305,  0.4897,
          0.6535,  0.3898, -0.6180, -1.3546, -1.5111,  0.5044, -0.6286, -0.2050]])
Generated geometry parameters from random latent vector: [ 0.13992532  0.0774395   0.20582182  0.04649636  0.23882247  0.19792212
  0.24291536  0.21831002  0.10978749  0.19874357 -0.13343595 -0.09857433
 -0.20612287 -0.05619303 -0.29143629 -0.17778978 -0.16126205 -0.27568779
  0.0676347   0.01091395  0.14532096  0.09997117  0.1989404   0.14596741
  0.11123403  0.37995635  0.12357761  0.34378344  0.10245465  0.39805643
 -0.15432585 -0.0893165  -0.24440596  0.02080222 -0.4373675  -0.04385582
 -0.29129174 -0.19184931  0.03810884  0.19596612 -1.02929401  0.772332
  0.60562587  0.62970809  0.02334017  0.9258633 ]


In [32]:
# 构造字典给ggs用
paramDict = {
    "init":
    {
        "upper_cst_params_root" : restored_generated_geometry[:10].tolist(),
        "lower_cst_params_root" : restored_generated_geometry[10:20].tolist(),
        "upper_cst_params_tip" : restored_generated_geometry[20:30].tolist(),
        "lower_cst_params_tip" : restored_generated_geometry[30:40].tolist(),
        "span" : pow(10,restored_generated_geometry[41].item()),
        "chord_root" : 1,
        "chord_tip" : pow(10,restored_generated_geometry[40].item()),
        "sweep" : math.atan(restored_generated_geometry[42].item()/restored_generated_geometry[43].item())*180/math.pi,  # 计算sweep角，单位为度
        "washout" : math.atan(restored_generated_geometry[44].item()/restored_generated_geometry[45].item())*180/math.pi
    }
}

In [33]:
SCRIPT_DIR = "D:/generate_3/Wing_generation/src"
paramDict = param_to_ggs_wing_dict(paramDict)
ggs_params = {
    'wings':[
                paramDict
            ]
}

generator.generate_script(
            script_file_path = os.path.join(SCRIPT_DIR, "script.vspscript"),
            global_params = ggs_params.get('global', None),
            fuselage = ggs_params.get('fuselage', None),
            wings = ggs_params.get('wings', None)
            )